In [ ]:
#Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#uploading data
from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
#load data into dataframe
data = pd.read_excel('iFood.xlsx')

In [ ]:
#load first_five rows
df=data
first_five = df.head(5)
print(first_five)

In [ ]:
"numwrical variables"
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

print("Numerical variables:", len(num_cols))
print("Categorical variables:", len(cat_cols))

In [ ]:
"number of variables"
print("Total variables:", len(df.columns))


In [ ]:
df.shape

In [ ]:
#dataset information
df.info()

In [ ]:
df.describe()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
# Check duplicate records
duplicates = df.duplicated().sum()
print("Number of duplicate records:", duplicates)

if duplicates == 0:
    print("No duplicates found. Dataset is clean.")
else:
    df = df.drop_duplicates()
    print("Duplicates removed.")

In [ ]:
# Check for outliers
sns.boxplot(x=df['Income'])
plt.show()

I used a boxplot was to assess the distribution of customer income and identify potential outliers. No significant outliers were observed, indicating that income values were within an acceptable range for analysis.

In [ ]:

sns.boxplot(x=df['MntTotal'])
plt.show()

A boxplot was used to illustrates the distribution of total customer expenditure



In [ ]:
df['MntTotal'].describe()

In [ ]:
df.nlargest(5, 'MntTotal')[['Income','MntTotal']]

In [ ]:
# check for need of starndadization
df['education'].unique()

In [ ]:
df['marital_status'].unique()

In [ ]:
#FEATURE ENGINEERING
# Total Purchases
df['TotalPurchases'] = (
    df['NumWebPurchases'] +
    df['NumCatalogPurchases'] +
    df['NumStorePurchases']
)

In [ ]:
df['Responded'] = np.where(df['AcceptedCmpOverall'] > 0, 1, 0)

df['Responded'].value_counts()

In [ ]:
#EDA
#Campain response distribution
sns.countplot(data=df, x='Responded')
plt.title('Campaign Response Distribution')
plt.show()

In [ ]:
#Income distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Income'], bins=30)
plt.title('Distribution of Customer Income')
plt.show()

In [ ]:
#Income vs Total Spending
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='Income', y='MntTotal')
plt.title('Income vs Total Spending')
plt.show()

In [ ]:
#Age distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Age'], bins=20)
plt.title('Age Distribution')
plt.show()

In [ ]:
#Response by education
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='education', hue='Responded')
plt.title('Campaign Response by Education Level')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#feature selection
X = df[['Income', 'Age', 'MntTotal', 'TotalPurchases']]
y = df['Responded']


In [ ]:


#MACHINE LEARNING
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Model Selection: Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
#Predictions
y_pred = model.predict(X_test)

In [ ]:
#Model Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



In [ ]:
#DECISION TREE
#Import and Train the Decision Tree
from sklearn.tree import DecisionTreeClassifier
#create model
dt_model = DecisionTreeClassifier(random_state=42)
#train model
dt_model.fit(X_train, y_train)


In [ ]:
#Make Predictions
y_pred_dt = dt_model.predict(X_test)

In [ ]:
#Model Evaluation
# Model Evaluation
print("Decision Tree Results")
print("-" * 40)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))


In [ ]:
#Visualize the Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt



In [ ]:
#Check Feature Importance
feature_importance = pd.Series(dt_model.feature_importances_, index=X.columns)
feature_importance.sort_values(ascending=False)



In [ ]:
import pandas as pd

#Plot Feature Importance
plt.figure(figsize=(8,5))

# Convert Series to DataFrame for seaborn barplot
feature_importance_df = feature_importance.reset_index()
feature_importance_df.columns = ['Feature', 'Importance']
feature_importance_df = feature_importance_df.sort_values('Importance', ascending=False)

sns.barplot(
    data=feature_importance_df,
    x='Importance',
    y='Feature'
)
plt.title('Decision Tree Feature Importance')
plt.show()

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)


In [ ]:
#Evaluate Random Forest
y_pred_rf = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
#feature importance chart
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

In [ ]:
#Export dataset
df.to_csv("ifood_cleaned.csv", index=False)

In [ ]:
#Downloading to the pc
from google.colab import files
files.download("ifood_cleaned.csv")